In [2]:
# ============================================================
# NB04a_Slow_Growers (FIXED)
# Dividend-focused screen for high-yield, financially healthy
# companies. No inverted PEG gate.
#
# Reads:  silver_lynch_features, slow_grower_parameters.csv
# Writes: gold_slow_grower_candidates, gold_slow_grower_screen
# ============================================================

import uuid, numpy as np, pandas as pd
from datetime import datetime

RUN_ID = str(uuid.uuid4())
SCREEN_DATE = str(datetime.now().date())

# Load config
try:
    cfg = (spark.read.format("csv").option("header", "true")
           .load("Files/config/slow_grower_parameters.csv")
           .toPandas().set_index("parameter")["value"].to_dict())
except Exception:
    cfg = {"min_dividend_yield": 0.025, "min_dividend_years": 4,
           "max_payout_ratio": 0.85, "max_debt_to_equity": 1.0,
           "min_interest_coverage": 3.0, "min_roe_5y": 0.08,
           "eps_cagr_max": 0.15, "max_pe_acceptable": 50.0,
           "target_slow_grower_count": 30, "max_per_industry": 5}

def f(k, d):
    try: return float(cfg.get(k, d))
    except: return float(d)

MIN_YIELD   = f("min_dividend_yield", 0.025)   # 2.5% yield floor
MIN_YEARS   = int(f("min_dividend_years", 4))  # 4 years of dividend history
MAX_PAYOUT  = f("max_payout_ratio", 0.85)     # Max 85% payout
MAX_DE      = f("max_debt_to_equity", 1.0)     # D/E <= 1.0
MIN_COV     = f("min_interest_coverage", 3.0)  # Coverage >= 3.0x
MIN_ROE     = f("min_roe_5y", 0.08)            # ROE >= 8%
MAX_CAGR    = f("eps_cagr_max", 0.15)          # Exclude growth above 15%
MAX_PE      = f("max_pe_acceptable", 50.0)
TARGET      = int(f("target_slow_grower_count", 30))
MAX_IND     = int(f("max_per_industry", 5))

# Load features
df = spark.table("silver_lynch_features").toPandas()
if df.empty:
    raise RuntimeError("silver_lynch_features is empty. Run NB03 first.")

# Add audit tracking columns to df FIRST
df["screen_date"] = SCREEN_DATE
df["run_id"] = RUN_ID

# Gates
df["g_yield"]       = df["ttm_dividend_yield"].notna() & (df["ttm_dividend_yield"] >= MIN_YIELD)
df["g_years"]       = df["dividend_years_10"].notna() & (df["dividend_years_10"] >= MIN_YEARS)
df["g_payout"]      = df["fcf_payout_3y"].isna() | (df["fcf_payout_3y"] <= MAX_PAYOUT)
df["g_debt"]        = df["debt_to_equity"].isna() | (df["debt_to_equity"] <= MAX_DE)
df["g_coverage"]    = df["interest_coverage"].isna() | (df["interest_coverage"] >= MIN_COV)
df["g_roe"]         = df["min_roe_4y"].notna() & (df["min_roe_4y"] >= MIN_ROE)
df["g_growth"]      = df["eps_cagr_3y"].isna() | (df["eps_cagr_3y"] <= MAX_CAGR)
df["g_pe_cap"]      = df["pe_current"].isna() | (df["pe_current"] <= MAX_PE)

slow_gates = ["g_yield", "g_years", "g_payout",
              "g_debt", "g_coverage", "g_roe", "g_growth", "g_pe_cap"]

# Print Diagnostics so you can see gate counts
print("=== SLOW GROWER GATE PASS COUNTS ===")
for g in slow_gates:
    print(f"  {g:<15}: {df[g].sum()} stocks passed")

df["all_slow_gates_pass"] = df[slow_gates].all(axis=1)
df["final_status"] = df["all_slow_gates_pass"].map(
    {True: "SLOW_GROWER_BUY", False: "SLOW_GROWER_WATCH"}
)

def reason(row):
    if row["all_slow_gates_pass"]:
        return "All slow grower gates passed"
    failed = [g for g in slow_gates if not row[g]]
    names = {"g_yield": f"yield<{MIN_YIELD*100:.1f}%",
             "g_years": f"div_years<{MIN_YEARS}",
             "g_payout": f"payout>{MAX_PAYOUT*100:.0f}%",
             "g_debt": f"D/E>{MAX_DE}",
             "g_coverage": f"int_cov<{MIN_COV}x",
             "g_roe": f"min_ROE<{MIN_ROE*100:.0f}%",
             "g_growth": f"growth>{MAX_CAGR*100:.0f}%",
             "g_pe_cap": f"P/E>{MAX_PE}"}
    return "Watch: failed " + ", ".join(names[g] for g in failed)

df["status_reason"] = df.apply(reason, axis=1)

# Dividend ranking score
df["dividend_score"] = (
    df["ttm_dividend_yield"].fillna(0) * 100 * 3
    + df["dividend_years_10"].fillna(0) * 2
    + (1 - df["debt_to_equity"].fillna(0.5).clip(upper=1.0)) * 20
)

# Save screens (every stock)
screen_cols = ["symbol", "company_name", "industry", "rule_set",
              "ttm_dividend_yield", "dividend_years_10", "dividend_no_cut_last5",
              "fcf_payout_3y", "debt_to_equity", "interest_coverage",
              "min_roe_4y", "eps_cagr_3y", "pe_current",
              "all_slow_gates_pass", "final_status", "status_reason", "screen_date", "run_id"]

spark.createDataFrame(df[screen_cols]).write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true").saveAsTable("gold_slow_grower_screen")

# Candidates selection
candidates = df[df["final_status"] == "SLOW_GROWER_BUY"].copy()

if not candidates.empty:
    candidates = candidates.sort_values("dividend_score", ascending=False)
    candidates["industry_rank"] = candidates.groupby("industry").cumcount() + 1
    candidates = candidates[candidates["industry_rank"] <= MAX_IND].head(TARGET)
    candidates["candidate_rank"] = range(1, len(candidates) + 1)
    candidates["candidate_type"] = "SLOW_GROWER"

    cand_cols = ["candidate_rank", "candidate_type", "symbol", "company_name",
                 "industry", "ttm_dividend_yield", "dividend_years_10",
                 "fcf_payout_3y", "debt_to_equity", "min_roe_4y",
                 "dividend_score", "status_reason", "screen_date", "run_id"]
                 
    spark.createDataFrame(candidates[cand_cols]).write.format("delta") \
        .mode("overwrite").option("overwriteSchema", "true") \
        .saveAsTable("gold_slow_grower_candidates")
        
    print(f"\n✅ SUCCESS: Saved gold_slow_grower_candidates ({len(candidates)} stocks)")
    display(candidates[["candidate_rank", "symbol", "company_name",
                        "ttm_dividend_yield", "dividend_years_10", "dividend_score"]])
else:
    empty_cand = pd.DataFrame(columns=["candidate_rank", "candidate_type", "symbol", "company_name",
                                       "industry", "ttm_dividend_yield", "dividend_years_10",
                                       "fcf_payout_3y", "debt_to_equity", "min_roe_4y",
                                       "dividend_score", "status_reason", "screen_date", "run_id"])
    spark.createDataFrame(empty_cand).write.format("delta").mode("overwrite") \
        .option("overwriteSchema", "true").saveAsTable("gold_slow_grower_candidates")
    print("\nNo slow growers met criteria.")

StatementMeta(, 54420ae8-2bb0-4b77-b1d3-26ba69a06680, 4, Finished, Available, Finished, False)

=== SLOW GROWER GATE PASS COUNTS ===
  g_yield        : 26 stocks passed
  g_years        : 238 stocks passed
  g_payout       : 282 stocks passed
  g_debt         : 285 stocks passed
  g_coverage     : 267 stocks passed
  g_roe          : 179 stocks passed
  g_growth       : 178 stocks passed
  g_pe_cap       : 237 stocks passed

✅ SUCCESS: Saved gold_slow_grower_candidates (4 stocks)


SynapseWidget(Synapse.DataFrame, fb1b3302-4ce7-4398-964b-a7553a26b88f)